# exp_1 — SetSeq-BD: set-sequence network with boundary-aware decoding

**Question.** Can a model that reads the *raw, identity-free player set* through time recover
possession across long ball gaps better than (a) hand-crafted rules and (b) hand-crafted features
with a gradient-boosted classifier — on the same frozen, degraded proxy?

**What is new here (not done in exp_0, exp_rasya or exp_ryu):**

| component | exp_1 | closest prior work in the team |
|---|---|---|
| input | DeepSets encoder over every visible player token (no identities, no hand features) | exp_0: 88 hand features; ryu: 23 hand team-shape/IFAB features; rasya: nearest-player rules |
| temporal model | 2-layer BiGRU over 60 s windows with a 3-class state head | ryu: dilated TCN (dead + team heads); exp_0: per-frame GBM; rasya: rule hold/carry |
| decoder | **boundary-aware Viterbi**: a learned boundary head sets a time-varying switch cost | exp_0 / ryu / rasya: one constant switch penalty (ryu: calibrated to a possession rate) |
| training data | **corruption resampling** of IDSSE (provider frame labels): each training match re-degraded with 2 extra seeds + home/away mirror | exp_0: one frozen corruption; ryu: Metrica + SkillCorner with event-derived labels; rasya: clean IDSSE |

ryu's work lives on the `ryu` git branch (the `exp_ryu/` working tree only holds the baseline); its shipped
model is run zero-shot on the proxy as the `ryu_final` reference arm in notebook 01.

**Protocol.** Seven IDSSE matches degraded by exp_0's frozen degrader (snapshot in `judge/`, SHA-256
checked). Nested leave-one-match-out: decoder settings for held-out match *h* are chosen on the
other six using models that never saw *h* or the validation match. Paired bootstrap over matches.
The reference arms (baseline, rasya C06/C08, exp_0 LightGBM) are re-run on the *identical* data.

**Data hygiene.** BAR-ATH is only touched in notebook 04, locally, with aggregate outputs only.
The public ground truth is opened once at the very end, after the predictor is frozen.


In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp1_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_1')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)
import exp1_common as C


## 1. Freeze the judge
The degrader and feature code are verbatim copies of exp_0 at the time of the snapshot. The copied degraded matches must be reproduced bit-for-bit by applying the frozen degrader to the clean matches.

In [2]:
judge = C.verify_judge()
proxy = C.verify_frozen_proxy()
C.write_json(C.RESULTS / 'judge_freeze.json', {'hashes': judge, 'frozen_proxy': proxy})
print(json.dumps(judge, indent=1))
print(pd.DataFrame(proxy).to_string(index=False))

{
 "judge/experiment_data.py": "e2b4034baa721ec7f0faf820626e6bb164325b55bf44595607612ab4c2cff7f1",
 "judge/experiment_model.py": "243dfe93ba4aa051d271bd59f21091c972abdf02baae58ac37423139b5333fbd",
 "official_metric": "341bad48904b489d6e875397b27687f5d6a4d22a397f3dc8af3a00d48f16874d"
}
 match  frozen_copy_matches_degrader
J03WMX                          True
J03WN1                          True
J03WOH                          True
J03WOY                          True
J03WPY                          True
J03WQQ                          True
J03WR9                          True


## 2. Protocol, frozen before any model is fitted

In [3]:
import exp1_net as N, exp1_decode as D, exp1_protocol as P
protocol = {
    'matches': C.MATCH_IDS,
    'evaluation_corruption_seed': 'BASE_SEED + index (exp_0 frozen)',
    'augmentation_rounds': C.AUG_ROUNDS,
    'augmentation_seed': 'BASE_SEED + 1000 * round + index (training matches only)',
    'model_fps': 5,
    'setseq_train_config': N.TrainConfig().as_dict(),
    'decoder_lambdas': D.LAMBDAS,
    'decoder_gammas': D.GAMMAS,
    'selection_objective': 'mean spellF1 - 0.15 * mean excess |count error| above 0.30 (exp_0)',
    'selection_scope': 'inner validation = six non-held-out matches, predicted by pair models trained without h and v',
    'bootstrap': '1000 paired match draws, seed 20261108, 95% percentile',
    'quality_gates': C.QUALITY_GATES,
    'model_variants': {k: list(map(str, v)) for k, v in P.DECODER_VARIANTS.items()},
    'rule_arms': P.RULE_ARMS,
}
path = C.RESULTS / 'protocol.json'
if path.exists() and json.loads(path.read_text()) != json.loads(json.dumps(protocol, default=str)):
    raise RuntimeError('Protocol changed after freezing; diagnose before rerunning')
C.write_json(path, protocol)
print('protocol sha256', C.sha256(path))

protocol sha256 3ad22aa57822db2aa76151b57816fd640efdf083cf24d55998ca067dfa58877e


## 3. Token cache: frozen corruption (round 0) + two augmentation rounds per match
Only aggregate statistics are shown. Augmentation rounds should look like round 0 statistically.

In [4]:
t0 = time.time()
summary = pd.DataFrame(P.build_token_cache())
summary.to_csv(C.RESULTS / 'data_summary.csv', index=False)
cols = ['match', 'round', 'ball_visible', 'gap_mean_seconds', 'mean_objects', 'tokens_per_step', 'boundary_rate', 'label_H', 'label_A', 'label_D']
print(summary[cols].round(3).to_string(index=False))
print('\nmean by round'); print(summary.groupby('round')[cols[2:]].mean().round(3).to_string())
print(f'built in {time.time() - t0:.0f}s')

 match  round  ball_visible  gap_mean_seconds  mean_objects  tokens_per_step  boundary_rate  label_H  label_A  label_D
J03WMX      0          0.59            16.978        20.484           20.490          0.061    0.268    0.317    0.415
J03WMX      1          0.59            16.978        20.474           20.469          0.061    0.268    0.317    0.415
J03WMX      2          0.59            16.978        20.437           20.435          0.061    0.268    0.317    0.415
J03WN1      0          0.59            16.946        19.557           19.557          0.061    0.257    0.226    0.517
J03WN1      1          0.59            16.946        19.512           19.516          0.061    0.257    0.226    0.517
J03WN1      2          0.59            16.946        19.580           19.579          0.061    0.257    0.226    0.517
J03WOH      0          0.59            17.047        20.447           20.434          0.054    0.285    0.239    0.476
J03WOH      1          0.59            17.047   

## 4. Sanity checks
* the mirror augmentation is an involution and swaps only H/A;
* the set encoder is invariant to token order;
* tokens do not depend on labels (shuffling labels leaves them unchanged).

In [5]:
import torch, exp1_tokens as T
from dataclasses import replace
item = P.load_tokens(C.MATCH_IDS[0])
tok, glo, st = item['tokens'][:300].astype('float32'), item['glob'][:300].astype('float32'), item['state'][:300]
t2, g2, s2 = T.mirror(*T.mirror(tok, glo, st))
print('mirror involution:', np.array_equal(t2, tok) and np.array_equal(g2, glo) and np.array_equal(s2, st))
print('mirror keeps D:', bool(np.all((T.mirror(tok, glo, st)[2] == 2) == (st == 2))))
torch.manual_seed(0)
model = N.SetSeq(64, 64).to(N.DEVICE).eval()
perm = np.random.default_rng(0).permutation(T.MAX_TOKENS)
args = lambda t, m: (torch.tensor(t)[None].to(N.DEVICE), torch.tensor(m)[None].to(N.DEVICE), torch.tensor(glo)[None].to(N.DEVICE))
with torch.no_grad():
    a = model(*args(tok, item['mask'][:300]))[0]
    b = model(*args(tok[:, perm], item['mask'][:300][:, perm]))[0]
print('token-order invariance, max |diff|:', float((a - b).abs().max()))
match = C.load_degraded(C.MATCH_IDS[0])
shuffled = replace(match, labels=np.random.default_rng(1).permutation(match.labels))
x1, x2 = T.build_tokens(match), T.build_tokens(shuffled)
print('label independence:', np.array_equal(x1.tokens, x2.tokens) and np.array_equal(x1.glob, x2.glob))
print('trainable parameters:', sum(p.numel() for p in model.parameters()))

mirror involution: True
mirror keeps D: True


token-order invariance, max |diff|: 2.944469451904297e-05


label independence: True
trainable parameters: 197317


## 5. exp_0 feature cache for the LightGBM reference arm

In [6]:
t0 = time.time()
P.build_gbm_features()
import exp1_arms as A
print({n: A.gbm_features(n)['x'].shape for n in C.MATCH_IDS}, f'{time.time() - t0:.0f}s')

{'J03WMX': (29194, 88), 'J03WN1': (28313, 88), 'J03WOH': (27443, 88), 'J03WOY': (28508, 88), 'J03WPY': (29243, 88), 'J03WQQ': (28470, 88), 'J03WR9': (29363, 88)} 0s
